# Week 1: Domain Understanding & Taxonomy — Data Exploration

**Goal:** understand what real MLS listing remarks look like *before* building the taxonomy, the labeled queries, and the cleaning pipeline.

**Inputs**
- `data/processed/listing_sample.csv` — 1,000 random listings from `rets_property` (`remarks` = `L_Remarks`). Created by `scripts/data_loading.py`. MLS data is git-ignored and not present in the repo.
- `data/processed/taxonomy.json` — 8-category real estate taxonomy, created by `scripts/taxonomy_builder.py`.
- `data/processed/sample_queries.json` — 60 hand-labeled search queries.

**Reproducibility:** the sample is a random draw (`ORDER BY RAND()`), so re-running `data_loading.py` produces slightly different numbers. The numbers quoted in the markdown cells come from the sample used for this run.


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
print("Repo root:", ROOT)
print("Files in data/processed:", sorted(p.name for p in DATA.iterdir()))

## 1. Load & Check sample

In [ ]:
df = pd.read_csv(DATA / "listing_sample.csv")
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df["remarks"].str.len().describe())

**What this shows**
- 1,000 rows, 7 columns. Nulls are negligible (address 4, city 3, beds 1); `remarks` has none because the SQL query filters on `L_Remarks IS NOT NULL AND LENGTH > 50`.
- Remarks average about 1,320 characters (median about 1,260), so these are full paragraphs, not one-liners. There is plenty of text per listing to extract from.


In [ ]:
print(df[["beds", "baths", "price"]].describe())

suspicious = pd.Series({
    "beds == 0": int((df["beds"] == 0).sum()),
    "baths == 0": int((df["baths"] == 0).sum()),
    "price > $10M": int((df["price"] > 10_000_000).sum()),
    "baths > 8": int((df["baths"] > 8).sum()),
})
print("\nSuspicious / outlier values:")
print(suspicious)

**Data Quality**
- A handful of listings report 0 beds or 0 baths. These may be land or data-entry gaps, so structured fields should not be trusted blindly when validating extracted entities.
- Price and bath counts have a long right tail (max price $45M, max baths 13). Medians are much more representative than means here.


## 2. Vocabulary

In [ ]:
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords"]:  
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
all_text = " ".join(df["remarks"].dropna().str.lower())
tokens = nltk.word_tokenize(all_text)
#keep alphabetic, non-stopword tokens only
tokens_clean = [t for t in tokens if t.isalpha() and t not in stop_words]

unigram_freq = Counter(tokens_clean)
for word, count in unigram_freq.most_common(60):
    print(f"{word}: {count}")

In [ ]:
from nltk.util import ngrams

bigram_freq = Counter(ngrams(tokens_clean, 2))
for bigram, count in bigram_freq.most_common(40):
    print(f"{' '.join(bigram)}: {count}")

- The most common words (`home`, `offers`, `features`, `opportunity`, `perfect`) are marketing filler. They are frequent but carry no searchable meaning.
- The real signal is in layout, kitchen and outdoor vocabulary: `primary suite`, `floor plan`, `stainless steel`, `granite/quartz countertops`, `soaking tub`, `attached garage`, `pool`.
- Some bigrams are artifacts of stopword removal, not real phrases (`pool spa` from "pool and spa", `shopping dining`, `living entertaining`).


In [ ]:
FILLER = [
    "thoughtfully designed", "rare opportunity", "conveniently located",
    "everyday living", "step outside", "easy access", "home offers",
    "highly sought after", "peace of mind", "perfect for entertaining",
]
lower = df["remarks"].str.lower()
filler_share = pd.Series(
    {p: lower.str.contains(p, regex=False).mean() * 100 for p in FILLER}
).sort_values(ascending=False)
print("Share of remarks containing each filler phrase (%):")
print(filler_share.round(1))

## 3. Formatting Patterns


In [ ]:
#prints raw remark text
for remark in df["remarks"].sample(5, random_state=42):
    print(remark[:400])
    print("---")

In [ ]:
html_count = df["remarks"].str.contains("<", regex=False).sum()
print(f"Remarks with possible HTML tags: {html_count}")

In [ ]:
patterns = {
    "curly quotes (\u2019 \u2018 \u201c \u201d)": r"[\u2019\u2018\u201c\u201d]",
    "embedded newlines": r"\n",
    "markdown asterisks (**)": r"\*\*",
    "ALL-CAPS words (4+ letters)": r"\b[A-Z]{4,}\b",
    "sq ft shorthand (sf, sqft, sq ft)": r"(?i)\b(?:sf|sqft|sq\.? ?ft)\b",
    "'&' instead of 'and'": r"&",
    "'approx' shorthand": r"(?i)\bapprox\b",
    "'w/' shorthand": r"\bw/",
    "dollar amounts ($123)": r"\$\d",
    "street address inside text": r"\b\d{3,5} [A-Z][a-z]+ (?:St|Street|Ave|Avenue|Dr|Drive|Rd|Road|Ln|Lane|Way|Ct|Court|Pl|Place|Blvd)\b",
    "HTML tags (<)": r"<",
}
rows = []
for name, pat in patterns.items():
    n = int(df["remarks"].str.contains(pat, regex=True).sum())
    rows.append((name, n, round(100 * n / len(df), 1)))
pd.DataFrame(rows, columns=["pattern", "remarks", "pct_of_sample"]).sort_values(
    "remarks", ascending=False).reset_index(drop=True)

In [ ]:
lengths = df["remarks"].str.len()
bins = [0, 250, 500, 1000, 1500, 2000, 3000, 4001]
print(pd.cut(lengths, bins=bins).value_counts().sort_index())
print("\nRemarks at exactly 4000 characters:", int((lengths == 4000).sum()))

**Formatting issues found**
- Smart quotes in about a quarter of remarks, and embedded newlines in about a quarter, so unicode and whitespace normalization are needed.
- Shorthand: sq ft variants (`sf`, `sqft`, `sq ft`) in roughly 18% of remarks, `&` in about 14%, plus occasional `approx.` and `w/`. Specific one-off abbreviations (`T&G`, `HWH`) also appear.
- About 3% contain stray markdown (`**`), and about 28% contain ALL-CAPS words, which affects case-sensitive matching.
- Street addresses appear inside remark text in roughly 8% of listings, which matters for entity extraction (an address number is not a bedroom count).
- No HTML tags in this sample, but the profiling report should still check for them.
- Maximum length is exactly 4,000 characters, which matches the 4000-character limit documented for `PublicRemarks` in the Trestle metadata. Only about 1 in 1,000 remarks hits it, so truncation is rare.


## 4. Taxonomy

In [ ]:
with open(DATA / "taxonomy.json") as f:
    terms = json.load(f)["terms"]
tax = pd.DataFrame(terms)
print(f"{len(tax)} terms in {tax['category'].nunique()} categories")
print(tax["category"].value_counts())

In [ ]:
def normalize(text):
    """Lowercase and treat hyphens as spaces: 'walk in closet' == 'walk-in closet'."""
    return re.sub(r"\s+", " ", str(text).lower().replace("-", " ")).strip()

def term_regex(term):
    #\b word boundaries so 'spa' does not match 'spacious'
    return re.compile(r"\b" + re.escape(normalize(term)) + r"\b")

texts = df["remarks"].dropna().map(normalize).tolist()
tax["pattern"] = tax["term"].map(term_regex)
tax["remarks_with_term"] = tax["pattern"].map(lambda p: sum(1 for t in texts if p.search(t)))

has_any = [any(p.search(t) for p in tax["pattern"]) for t in texts]
print(f"Overall coverage: {sum(has_any) / len(texts):.1%} of remarks contain at least one taxonomy term")

In [ ]:
prevalence = {}
for category, group in tax.groupby("category"):
    patterns_ = list(group["pattern"])
    prevalence[category] = sum(1 for t in texts if any(p.search(t) for p in patterns_)) / len(texts)
print("Share of remarks mentioning each category (%):")
print((pd.Series(prevalence).sort_values(ascending=False) * 100).round(1))

In [ ]:
print("Top 15 terms by number of remarks:")
print(tax.sort_values("remarks_with_term", ascending=False)[["term", "category", "remarks_with_term"]]
      .head(15).to_string(index=False))

zero = tax[tax["remarks_with_term"] == 0]
print(f"\nZero-hit terms ({len(zero)}):", list(zero["term"]))

**Taxonomy findings**
- 8 categories, 379 terms. Coverage is far above the 30% requirement, and every category appears in the sample.
- `financing_ownership` is the thinnest category in the remarks (about a quarter of listings), mostly HOA, rental-income and investor language. Lender-style terms (conventional loan, foreclosure, bank owned) were absent in 1,000 remarks and were pruned.
- The one deliberately kept zero-hit term, `near university`, supports query-side phrases like "near UC Irvine".


## 5. Labeled queries

In [ ]:
with open(DATA / "sample_queries.json") as f:
    q = pd.DataFrame(json.load(f))

q["words"] = q["query"].str.split().str.len()
q["has_taxonomy_term"] = q["query"].map(lambda s: any(p.search(normalize(s)) for p in tax["pattern"]))

print(q["intent"].value_counts())
print()
print(q.groupby("intent")[["words", "has_taxonomy_term"]].mean().round(2))

**Query set**
- 60 queries, 20 each for `browsing`, `researching`, `high_intent_inquiry` (the same labels Week 7 will train on).
- Labeling rules: *browsing* = broad and unconstrained; *researching* = area, cost or comparison questions; *high intent* = specific constraints plus readiness or availability.
- Cities come from the sample itself, and several queries are deliberately messy (`3br 2ba`, `w/`, `$1.2M`, `&`) so they double as Week 2 normalization test inputs.

## Summary
1. **Data:** 1,000-remark sample loaded; structured fields mostly clean, with a few 0-bed/0-bath rows and extreme prices.
2. **Taxonomy:** 379 terms across 8 categories, 99%+ coverage, validated by `tests/test_week1.py`.
3. **Queries:** 60 labeled queries, balanced across three intents.
4. **Week 2 inputs:** smart quotes, newlines, ALL-CAPS, sq ft shorthand, `&`, stray markdown, embedded addresses, filler phrases, and plural/hyphen variants.
